<a id="top"></a>

# Lab L1004: build the ReAct agent graph (termination)

```yaml
title: "Lab L1004: build the ReAct agent graph (termination)"
keywords: ReAct agent, StateGraph, add_messages, ToolNode, route, conditional edge, back-edge, cycle, natural termination, recursion_limit, GraphRecursionError, FakeModel, offline, lab
estimated duration: 40
```

> **Lesson:** L10. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L10/objectives.md).
> Student copy — fill in every `# TODO`. Other copy: `L1004_lab_solutions.ipynb`. **Pure Python — no API key needed.** You wire
> the graph against a *scripted stub model*, so it runs the same way every time (the same offline
> approach as the [L1003 demo](L1003_lecture.ipynb)).
>
> **After this lab you can:** wire a ReAct agent as a cyclic `StateGraph` — an `agent` node, a
> prebuilt `ToolNode`, a `route` conditional edge, and a **back-edge** — and reason about how it
> terminates: naturally when `route` returns `END`, or on the `recursion_limit` cap when it doesn't.

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 — Write route (the conditional edge)](#2-problem-1--write-route-the-conditional-edge)
- [3. Problem 2 — Wire the graph: build_agent](#3-problem-2--wire-the-graph-build_agent)
- [4. Problem 3 — Drive it: natural termination](#4-problem-3--drive-it-natural-termination)
- [5. Problem 4 — Drive it: the recursion_limit catches a runaway](#5-problem-4--drive-it-the-recursion_limit-catches-a-runaway)
- [6. Problem 5 — The message-history invariant (written)](#6-problem-5--the-message-history-invariant-written)
- [7. Self-check](#7-self-check)

## 1. Setup

All given: the `calculator` + `lookup` tools and the `TOOLS` list; a scripted **`FakeModel`** (from
`common`) that mimics a LangChain chat model — `.bind_tools(...)` then `.invoke(messages)` returns
the next scripted `AIMessage`; and the **`AgentState`** with its `add_messages` reducer. You write
the graph: `route`, then `build_agent`.

In [ ]:
from typing import Annotated, Any, TypedDict

from langchain_core.messages import AIMessage, BaseMessage, HumanMessage
from langgraph.errors import GraphRecursionError
from langgraph.graph import END, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

from fluffy_potato_curriculum.common.fake_model import (
    FakeModel,
    text_reply,
    tool_call,
    tool_reply,
)


def calculator(expression: str) -> str:
    """Evaluate a simple arithmetic expression, or raise ValueError on non-arithmetic input."""
    allowed = set("0123456789+-*/(). ")
    if not expression or set(expression) - allowed:
        raise ValueError(f"refusing to evaluate non-arithmetic expression: {expression!r}")
    return str(eval(expression))


POPULATIONS = {"tokyo": "37000000", "lagos": "15000000", "paris": "11000000"}


def lookup(city: str) -> str:
    """Return a city's population, or raise KeyError if it isn't in the table."""
    key = city.strip().lower()
    if key not in POPULATIONS:
        raise KeyError(f"no population on file for {city!r}")
    return POPULATIONS[key]


# The tool list handed to both the model (bind_tools) and the ToolNode.
TOOLS = [calculator, lookup]


class AgentState(TypedDict):
    """The agent's state. The `add_messages` reducer APPENDS each node's messages to the
    running conversation (in L04 a returned key OVERWROTE it) -- that is why the history grows."""

    messages: Annotated[list[BaseMessage], add_messages]

[↑ Back to top](#top)

## 2. Problem 1 — Write `route` (the conditional edge)

Warm-up before the full graph. `route(state)` is the L05 conditional edge: look at the **last**
message. If it's an `AIMessage` whose `.tool_calls` is non-empty, return `"tools"` (run them);
otherwise return `END` — the model sent plain text, which is *natural termination*.

In [ ]:
def route(state: AgentState) -> str:
    """Conditional edge: got tool calls? -> the 'tools' node; else -> END (natural termination)."""
    # TODO: look at the LAST message, state["messages"][-1].
    # If it is an AIMessage whose .tool_calls is non-empty, return "tools".
    # Otherwise return END.
    raise NotImplementedError


# Quick checks (they should print "tools" then "__end__" once route is implemented):
print(route({"messages": [tool_reply(tool_call("c1", "lookup", {"city": "Paris"}))]}))
print(route({"messages": [text_reply("all done")]}))

[↑ Back to top](#top)

## 3. Problem 2 — Wire the graph: `build_agent`

Now the whole graph. Fill in `agent_node` (bind `TOOLS`, `.invoke(state["messages"])`, return
`{"messages": [reply]}`), then wire and compile: `StateGraph(AgentState)`, add the `agent` node and
a `ToolNode(TOOLS)` `tools` node, `set_entry_point("agent")`, `add_conditional_edges("agent", route,
{"tools": "tools", END: END})`, and the **back-edge** `add_edge("tools", "agent")`.

In [ ]:
def build_agent(model: Any) -> Any:
    """Wire and compile the ReAct graph: agent node + ToolNode + conditional route + back-edge."""

    def agent_node(state: AgentState) -> dict[str, list[BaseMessage]]:
        """Call the tool-bound model on the conversation; return its one reply to be appended."""
        # TODO: bind TOOLS to the model, invoke it on state["messages"], and return the reply
        # wrapped as {"messages": [reply]} so the add_messages reducer appends it.
        raise NotImplementedError

    # TODO: wire the graph.
    #   1. builder = StateGraph(AgentState)
    #   2. add_node("agent", agent_node) and add_node("tools", ToolNode(TOOLS))
    #   3. set_entry_point("agent")
    #   4. add_conditional_edges("agent", route, {"tools": "tools", END: END})
    #   5. add_edge("tools", "agent")   <- the BACK-EDGE that makes it a cycle
    #   6. return builder.compile()
    raise NotImplementedError


# Confirm the shape once build_agent is implemented:
rendered = build_agent(FakeModel([text_reply("x")])).get_graph()
print("nodes:", list(rendered.nodes))
print(
    "back-edge tools -> agent:",
    ("tools", "agent") in [(e.source, e.target) for e in rendered.edges],
)

[↑ Back to top](#top)

## 4. Problem 3 — Drive it: natural termination

Script a stub that calls `calculator`, then `lookup`, then answers in plain text. Build the graph,
`invoke` it, and confirm the tool sequence was `['calculator', 'lookup']` and the last message is a
plain-text `AIMessage` (no tool calls) — *natural termination*.

In [ ]:
# TODO: script a FakeModel that (1) calls calculator, (2) calls lookup, (3) answers in plain text.
#   happy_script = [
#       tool_reply(tool_call("c1", "calculator", {"expression": "36 * 1"})),
#       tool_reply(tool_call("c2", "lookup", {"city": "Lagos"})),
#       text_reply("Lagos has about 15,000,000 people."),
#   ]
# Then build_agent(FakeModel(happy_script)), invoke it on a task, and read result["messages"].
happy_script = ...  # TODO
graph = ...  # TODO: build_agent(FakeModel(happy_script))
task = {"messages": [HumanMessage(content="Population of Lagos? Use the tools.")]}
result = graph.invoke(task)

tool_sequence = [
    c["name"] for m in result["messages"] if isinstance(m, AIMessage) for c in m.tool_calls
]
print("tool sequence:", tool_sequence)
print("final answer :", result["messages"][-1].content)
assert tool_sequence == ["calculator", "lookup"]
assert isinstance(result["messages"][-1], AIMessage) and not result["messages"][-1].tool_calls

[↑ Back to top](#top)

## 5. Problem 4 — Drive it: the `recursion_limit` catches a runaway

Script a stub that **never stops** — many distinct `lookup` turns. Invoke with
`{"recursion_limit": 6}` and confirm a `GraphRecursionError` fires. This is the safety net: a cyclic
graph with no cap is a runaway waiting to happen, and hitting the cap is a signal worth
investigating.

In [ ]:
# TODO: build a runaway. Script SEVERAL DISTINCT lookup turns (fresh ids r0, r1, ...) so the
# add_messages reducer appends each one -- repeating the same object would be de-duplicated by id.
#   runaway_script = [tool_reply(tool_call(f"r{i}", "lookup", {"city": "Tokyo"})) for i in range(8)]
# Build the graph, then invoke with {"recursion_limit": 6} inside try/except GraphRecursionError,
# and set `raised = True` in the except block.
runaway_script = ...  # TODO
runaway_graph = ...  # TODO: build_agent(FakeModel(runaway_script))

raised = False
# TODO: invoke runaway_graph with {"recursion_limit": 6}; catch GraphRecursionError -> raised = True
raise NotImplementedError

print("recursion_limit fired:", raised, " <- the cap caught the runaway, not the model")
assert raised

[↑ Back to top](#top)

## 6. Problem 5 — The message-history invariant (written)

After the `agent` node returns an `AIMessage` carrying **two** tool calls, what must be true of the
message list *before* the next `agent` visit — and which **two prebuilt pieces** guarantee it
without you writing any bookkeeping? (Hint: one is the state's reducer; one is a node.)

_Write your answer by editing this cell (double-click)._

_TODO_

[↑ Back to top](#top)

## 7. Self-check

Compare against `L1004_lab_solutions.ipynb`. Done when: `route` returns `"tools"` for a tool-call
reply and `END` for a text reply; `build_agent` compiles a graph with an `agent` node, a `tools`
node, and a `tools → agent` back-edge; the happy run's tool sequence is `['calculator', 'lookup']`
and it terminates naturally; the runaway run raises `GraphRecursionError` at `recursion_limit=6`; and
you can name the two prebuilt pieces that maintain the message-history invariant.

[↑ Back to top](#top)